In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

plt.rcParams['figure.figsize'] = (12,5)
sns.set(style="whitegrid")


In [7]:
news_path = "../data/news_cleaned.csv"
df_news = pd.read_csv(news_path)

df_news["date"] = pd.to_datetime(df_news["date"], utc=True, errors="coerce")

df_news = df_news.dropna(subset=["headline"])
df_news.head()


,Unnamed: 0,headline,url,publisher,date,stock,headline_length,day_of_week,hour,publisher_domain
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 14:30:54+00:00,A,39,Friday,14.0,unknown
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 14:45:20+00:00,A,42,Wednesday,14.0,unknown
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 08:30:07+00:00,A,29,Tuesday,8.0,unknown
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 16:45:06+00:00,A,44,Friday,16.0,unknown
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 15:38:59+00:00,A,87,Friday,15.0,unknown


In [ ]:
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df_news["sentiment"] = df_news["headline"].apply(get_sentiment)
df_news[["headline", "sentiment"]].head()


In [ ]:
df_news["date_only"] = df_news["date"].dt.date
daily_sentiment = df_news.groupby("date_only")["sentiment"].mean().reset_index()
daily_sentiment.rename(columns={"sentiment": "avg_daily_sentiment"}, inplace=True)

daily_sentiment.head()


In [ ]:
import os

DATA_DIR = "../data/yfinance_data/Data"
symbol = "AAPL"

price_file = os.path.join(DATA_DIR, f"{symbol}.csv")
df_price = pd.read_csv(price_file)

date_col = [c for c in df_price.columns if c.lower() in ["date", "timestamp"]][0]
df_price[date_col] = pd.to_datetime(df_price[date_col], utc=True)
df_price = df_price.set_index(date_col)

df_price = df_price[["Open", "High", "Low", "Close", "Volume"]]
df_price.head()


In [ ]:
df_price["return"] = df_price["Close"].pct_change()

df_returns = df_price[["return"]].copy()
df_returns["date_only"] = df_returns.index.date

df_returns.head()


In [ ]:
df_merged = pd.merge(
    daily_sentiment,
    df_returns,
    left_on="date_only",
    right_on="date_only",
    how="inner"
)

df_merged.head()


In [ ]:
corr_value = df_merged["avg_daily_sentiment"].corr(df_merged["return"])
corr_value


In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df_merged,
    x="avg_daily_sentiment",
    y="return"
)
plt.title(f"AAPL: Daily Sentiment vs Daily Return (Correlation = {corr_value:.3f})")
plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Return")
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(14,6))

ax1.plot(df_merged["date_only"], df_merged["avg_daily_sentiment"], label="Sentiment", color="blue")
ax1.set_ylabel("Daily Avg Sentiment")

ax2 = ax1.twinx()
ax2.plot(df_merged["date_only"], df_merged["return"], label="Daily Return", color="red")
ax2.set_ylabel("Daily Return")

plt.title("Sentiment vs Stock Returns Over Time")
plt.show()


In [ ]:
df_merged.to_csv("../data/sentiment_return_merged.csv", index=False)
print("Saved merged dataset.")
